# 🍎 당도 센서 IoT 시뮬레이션 실습
### 파이썬으로 센서부터 클라우드까지 — 정보통신의 흐름을 직접 만들어 보기

장비 없이 파이썬만으로 **디지털 당도계(굴절계)** 가 측정한 값이 농장 관리자의 화면에 도착하기까지의 전 과정을 시뮬레이션합니다.
각 단계를 직접 코드로 만들면서 **파이썬 문법**과 **정보통신 개념**을 함께 익힙니다.

```
 [과즙]  →  ① 센서(빛의 굴절)  →  ② ADC(아날로그→디지털)  →  ③ 직렬통신 프레임(UART)
                                                                     ↓
 ⑥ 데이터 분석·시각화  ←  ⑤ MQTT 발행/구독(브로커)  ←  ④ 계층별 캡슐화(TCP/IP)
```

| 실습 | 주제 | 파이썬 개념 | 정보통신 개념 |
|---|---|---|---|
| 1 | 굴절계 센서 원리 | 함수, `math` 모듈, 반복문 | 센서, 물리량→전기신호, 분해능 |
| 2 | ADC와 잡음 | 리스트, `random`, 정수 연산 | 양자화, 비트 수, 잡음·SNR, 평균화 |
| 3 | 직렬통신 프레임 | `bytes`, `struct`, 비트 연산(`^`, `<<`) | 프레임 구조, 전송속도(bps), 체크섬·CRC, 비트오류율(BER) |
| 4 | 계층별 캡슐화 | `json`, 딕셔너리, 문자열 포매팅 | OSI/TCP-IP 계층, 헤더, 오버헤드, 전송 효율 |
| 5 | MQTT 발행/구독 | 클래스, 콜백 함수, 딕셔너리 | 브로커, 토픽·와일드카드, QoS, 패킷 손실·재전송 |
| 6 | 데이터 분석 | `pandas`, `matplotlib` | 데이터 수집·저장·시각화, 경보 |
| 7 | (선택) 실제 MQTT | 외부 라이브러리 `paho-mqtt` | 공개 브로커, 평문 통신의 보안 문제 |

**진행 방법**
- 위에서부터 차례로 셀을 실행하세요(`Shift + Enter`).
- ✏️ 표시는 직접 코드를 작성하는 과제입니다. 정답은 바로 아래 **▶ 정답 보기**를 눌러 확인합니다.
- 실습 7을 제외하면 인터넷 연결이나 장비가 필요 없습니다.

> ⚠️ 이 노트북의 굴절률 공식과 기준 당도 값은 **교육용 근사치**입니다. 실제 계측기 설계에 그대로 쓰면 안 됩니다.

## 0. 준비

In [ ]:
import math
import random
import struct
import json
import binascii
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 실행할 때마다 같은 결과가 나오도록 난수 시드를 고정합니다
random.seed(42)
np.random.seed(42)

# 그래프 스타일: 흑백 선 위주
INK, GRAY, LIGHT = "#222222", "#888888", "#cccccc"
plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True, "grid.color": "#e6e6e6",
    "axes.edgecolor": "#444444", "axes.prop_cycle": plt.cycler(color=[INK]),
})
print("준비 완료!")

> 💡 그래프 라벨은 한글 폰트 문제를 피하려고 영어로 표기했습니다. 한글 라벨을 쓰고 싶다면 아래 셀의 주석을 풀고 실행하세요.

In [ ]:
# !pip -q install koreanize-matplotlib
# import koreanize_matplotlib

---
## 실습 1. 굴절계 센서의 원리 — 물리량을 숫자로

### 📖 개념
설탕이 많이 녹은 과즙일수록 빛이 더 많이 꺾입니다(**굴절률 n**이 커짐). 디지털 굴절계는 과즙을 프리즘 위에 올리고,
빛이 **전반사되기 시작하는 각도(임계각 θc)** 를 측정합니다.

$$\sin\theta_c = \frac{n_{과즙}}{n_{프리즘}}$$

프리즘 아래에는 작은 광센서가 한 줄로 늘어선 **라인 센서**가 있어서, 밝은 영역과 어두운 영역의 **경계가 몇 번째 픽셀**에 생기는지를 읽습니다.

```
 당도(°Brix) → 굴절률 n → 임계각 θc → 경계 픽셀 번호   (측정)
 경계 픽셀 번호 → θc → n → 당도(°Brix)                 (계산으로 되돌리기)
```

**°Brix**: 용액 100g에 녹아 있는 당의 g 수. 10 °Brix ≈ 설탕 10g + 물 90g.

In [ ]:
N_PRISM = 1.72          # 프리즘 굴절률 (고굴절 광학유리 가정)

def brix_to_n(brix):
    # 당도(°Brix) → 굴절률 (20°C 기준 교육용 근사식)
    return 1.33299 + 0.00142 * brix + 0.0000045 * brix ** 2

def critical_angle_deg(n_sample, n_prism=N_PRISM):
    # 임계각(도) = asin(n_sample / n_prism)
    return math.degrees(math.asin(n_sample / n_prism))

print(" 당도(°Brix)   굴절률 n    임계각 θc")
for b in [0, 5, 10, 15, 20, 30]:
    n = brix_to_n(b)
    print(f"{b:>8}      {n:.5f}    {critical_angle_deg(n):.3f}°")

### 라인 센서 만들기
각도 범위 50.5°~54.5°를 256개 픽셀이 나눠서 봅니다. 임계각보다 큰 쪽은 전반사되어 **밝고**, 작은 쪽은 **어둡습니다**.

In [ ]:
PIXELS = 256
THETA_MIN, THETA_MAX = 50.5, 54.5     # 라인 센서가 보는 각도 범위(도)

def angle_to_pixel(theta):
    # 각도 → 픽셀 번호(0 ~ PIXELS-1)
    ratio = (theta - THETA_MIN) / (THETA_MAX - THETA_MIN)
    return max(0, min(PIXELS - 1, round(ratio * (PIXELS - 1))))

def pixel_to_angle(pixel):
    return THETA_MIN + pixel / (PIXELS - 1) * (THETA_MAX - THETA_MIN)

def edge_pixel(brix):
    # 당도에 해당하는 명암 경계 픽셀 번호
    return angle_to_pixel(critical_angle_deg(brix_to_n(brix)))

# 세 가지 당도에서 라인 센서가 보는 밝기
fig, ax = plt.subplots()
for brix, style in [(5, "-"), (15, "--"), (25, ":")]:
    e = edge_pixel(brix)
    light = [0.1 if i < e else 1.0 for i in range(PIXELS)]   # 경계 아래 어두움, 위 밝음
    ax.step(range(PIXELS), light, style, color=INK, label=f"{brix} Brix (edge={e})")
ax.set_xlabel("Pixel index"); ax.set_ylabel("Light intensity")
ax.set_title("Line sensor image vs sugar content"); ax.legend()
plt.show()

### ✏️ 과제 1-1. 픽셀 번호로 당도 계산하기
`pixel_to_brix(pixel)` 함수를 완성하세요. 순서: 픽셀 → 각도(`pixel_to_angle`) → 굴절률(`N_PRISM × sin θ`) → 당도(2차 방정식의 근).

힌트: $aB^2 + bB + c = 0$ 에서 $a=0.0000045,\ b=0.00142,\ c=1.33299-n$, 근은 $B = \dfrac{-b+\sqrt{b^2-4ac}}{2a}$

In [ ]:
def pixel_to_brix(pixel):
    # 여기에 코드를 작성하세요
    pass

# 확인용: 정답이면 원래 당도와 비슷한 값이 나옵니다
# for b in [5, 10, 15, 20]:
#     print(b, "→", round(pixel_to_brix(edge_pixel(b)), 2))

<details><summary>▶ 정답 보기</summary>

```python
def pixel_to_brix(pixel):
    theta = math.radians(pixel_to_angle(pixel))
    n = N_PRISM * math.sin(theta)
    a, b, c = 0.0000045, 0.00142, 1.33299 - n
    return (-b + math.sqrt(b * b - 4 * a * c)) / (2 * a)
```
</details>

In [ ]:
# 정답 코드 (다음 실습에서 사용하므로 반드시 실행하세요)
def pixel_to_brix(pixel):
    theta = math.radians(pixel_to_angle(pixel))
    n = N_PRISM * math.sin(theta)
    a, b, c = 0.0000045, 0.00142, 1.33299 - n
    return (-b + math.sqrt(b * b - 4 * a * c)) / (2 * a)

for b in [5, 10, 15, 20]:
    p = edge_pixel(b)
    print(f"실제 {b:>2} °Brix → 경계 픽셀 {p:>3} → 계산 {pixel_to_brix(p):.2f} °Brix")

# 분해능: 픽셀 1칸이 당도 몇 °Brix에 해당할까?
res = pixel_to_brix(edge_pixel(15) + 1) - pixel_to_brix(edge_pixel(15))
print(f"\n분해능(15 °Brix 부근): 픽셀 1칸 ≈ {res:.3f} °Brix")

### 🤔 생각해 보기
- 실제 당도와 계산 당도가 조금 다른 이유는? (힌트: `round`)
- `PIXELS`를 64나 1024로 바꾸고 다시 실행해 보세요. **분해능**은 어떻게 변하나요? 센서 픽셀이 많아지면 어떤 비용이 늘어날까요?

---
## 실습 2. ADC — 아날로그 전압을 디지털 숫자로

### 📖 개념
광센서는 빛의 세기를 **전압(아날로그)** 으로 냅니다. 마이크로컨트롤러는 **ADC(Analog-to-Digital Converter)** 로 이 전압을 정수로 바꿉니다.

- **n비트 ADC** → $2^n$ 단계로 표현 (8비트 = 256단계, 12비트 = 4096단계)
- 계단처럼 끊어서 표현하면서 생기는 오차 = **양자화 오차**
- 기준전압 3.3V, 12비트라면 한 단계 = 3.3 / 4095 ≈ 0.8mV

In [ ]:
VREF = 3.3

def adc(voltage, bits):
    # 전압(0~VREF) → 정수 코드(0 ~ 2^bits - 1)
    top = 2 ** bits - 1
    code = int(voltage / VREF * top + 0.5)       # 반올림
    return max(0, min(top, code))                # 범위를 벗어나면 잘라냄(clipping)

def code_to_voltage(code, bits):
    return code / (2 ** bits - 1) * VREF

print(" 비트   단계 수   1단계 전압(mV)   1.234V 변환 결과")
for bits in [3, 8, 10, 12]:
    c = adc(1.234, bits)
    print(f"{bits:>4} {2**bits:>9} {VREF/(2**bits-1)*1000:>14.2f}     코드 {c:>4} → {code_to_voltage(c, bits):.4f} V")

In [ ]:
# 서서히 올라가는 전압을 3비트와 6비트로 변환해 비교
v = np.linspace(0, VREF, 400)
fig, ax = plt.subplots()
ax.plot(v, v, color=LIGHT, lw=3, label="Analog (original)")
ax.plot(v, [code_to_voltage(adc(x, 3), 3) for x in v], color=INK, label="3-bit ADC")
ax.plot(v, [code_to_voltage(adc(x, 6), 6) for x in v], "--", color=GRAY, label="6-bit ADC")
ax.set_xlabel("Input voltage (V)"); ax.set_ylabel("Digital output (V)")
ax.set_title("Quantization: more bits, finer steps"); ax.legend()
plt.show()

### 잡음(Noise)과 평균화
실제 전압에는 항상 **잡음**이 섞입니다. 측정을 여러 번 해서 **평균**을 내면 잡음이 줄어듭니다(평균 횟수를 4배로 늘리면 잡음은 약 1/2).
라인 센서 전체를 ADC로 읽어 경계를 찾고 당도를 계산하는 **측정 함수**를 만들어 봅시다.

In [ ]:
V_BRIGHT, V_DARK = 2.8, 0.4      # 밝은 픽셀, 어두운 픽셀 전압

def read_line_sensor(brix, bits=12, noise=0.05, averages=1):
    # 라인 센서를 averages번 읽어 평균 → ADC 코드 리스트
    e = edge_pixel(brix)
    codes = []
    for i in range(PIXELS):
        ideal = V_DARK if i < e else V_BRIGHT
        volt = sum(ideal + random.gauss(0, noise) for _ in range(averages)) / averages
        codes.append(adc(volt, bits))
    return codes

def measure_brix(brix, bits=12, noise=0.05, averages=1):
    codes = read_line_sensor(brix, bits, noise, averages)
    threshold = (max(codes) + min(codes)) / 2
    dark_count = sum(1 for c in codes if c < threshold)   # 어두운 픽셀 수 = 경계 위치
    return pixel_to_brix(dark_count)

for noise in [0.05, 0.6]:
    print(f"잡음 {noise}V:", [round(measure_brix(15, noise=noise), 2) for _ in range(5)])

In [ ]:
# 실험: 잡음 크기 × 평균 횟수 → 측정 오차(표준편차)
rows = []
for noise in [0.3, 0.6, 0.9]:
    for avg in [1, 4, 16]:
        results = [measure_brix(15, noise=noise, averages=avg) for _ in range(150)]
        rows.append({"잡음(V)": noise, "평균 횟수": avg,
                     "평균 측정값": round(np.mean(results), 2),
                     "오차 표준편차": round(np.std(results), 3)})
pd.DataFrame(rows)

### 🤔 생각해 보기
- 잡음이 클 때 평균 횟수를 늘리면 오차가 어떻게 변하나요? 대신 무엇을 잃을까요? (힌트: 측정 시간)
- 잡음이 0.6V일 때 평균 측정값이 15에서 벗어나는(치우치는) 이유는 무엇일까요?
- `bits=8`로 바꿔서 실험해 보세요. 이 측정에서는 비트 수와 잡음 중 무엇이 더 중요한가요?

---
## 실습 3. 직렬통신 프레임 — 센서가 보내는 바이트 묶음

### 📖 개념
측정값은 센서 모듈에서 메인 보드로 **UART 직렬통신**으로 보냅니다. 수신 측이 데이터의 시작과 끝, 오류 여부를 알 수 있도록 정해진 형식의 **프레임**에 담습니다.

```
┌──────┬──────────┬──────┬────────────┬────────────┬──────────┬──────┐
│ STX  │ SENSOR_ID│ SEQ  │ BRIX × 10  │ TEMP × 10  │ CHECKSUM │ ETX  │
│ 0x02 │  1 byte  │1 byte│  2 bytes   │  2 bytes   │  1 byte  │ 0x03 │
└──────┴──────────┴──────┴────────────┴────────────┴──────────┴──────┘   총 9 바이트
```
- **STX/ETX**: 시작(Start of TeXt)과 끝(End of TeXt) 표시
- **SEQ**: 일련번호 — 빠진 프레임을 찾는 데 사용
- 소수점 값은 10을 곱해 **정수**로 보냄 (14.3 → 143)
- **체크섬**: 데이터 바이트를 모두 XOR한 값. 수신 측이 다시 계산해 비교

In [ ]:
STX, ETX = 0x02, 0x03

def xor_checksum(data: bytes) -> int:
    c = 0
    for b in data:
        c ^= b          # XOR(배타적 논리합) 누적
    return c

def build_frame(sensor_id, seq, brix, temp_c):
    # '>BBHh' = 빅엔디언, uint8, uint8, uint16, int16
    body = struct.pack(">BBHh", sensor_id, seq & 0xFF, round(brix * 10), round(temp_c * 10))
    return bytes([STX]) + body + bytes([xor_checksum(body), ETX])

def parse_frame(frame: bytes):
    if len(frame) != 9 or frame[0] != STX or frame[-1] != ETX:
        return None, "FRAME_ERR"
    body, chk = frame[1:7], frame[7]
    if xor_checksum(body) != chk:
        return None, "CHECKSUM_ERR"
    sid, seq, b10, t10 = struct.unpack(">BBHh", body)
    return {"sensor_id": sid, "seq": seq, "brix": b10 / 10, "temp": t10 / 10}, "OK"

f = build_frame(sensor_id=7, seq=1, brix=14.3, temp_c=21.5)
print("16진수:", f.hex(" ").upper())
print("2진수 :", " ".join(f"{b:08b}" for b in f[:4]), "...")
print("해석  :", parse_frame(f))

### 전송 시간 계산
UART는 1바이트를 보낼 때 **시작비트 1 + 데이터 8 + 정지비트 1 = 10비트**를 씁니다.

In [ ]:
for bps in [9600, 115200]:
    bits_total = len(f) * 10
    print(f"{bps:>6} bps: {len(f)}바이트 × 10비트 = {bits_total}비트 → {bits_total / bps * 1000:.2f} ms")

### 잡음 채널 — 비트가 뒤집힌다
전선이나 무선 구간에서는 0이 1로, 1이 0으로 뒤집히는 **비트 오류**가 생깁니다. 비트 하나가 뒤집힐 확률을 **BER(Bit Error Rate)** 라고 합니다.

In [ ]:
def noisy_channel(frame: bytes, ber: float) -> bytes:
    out = bytearray(frame)
    for i in range(len(out)):
        for bit in range(8):
            if random.random() < ber:
                out[i] ^= (1 << bit)      # 해당 비트를 뒤집기
    return bytes(out)

random.seed(1)
bad = noisy_channel(f, ber=0.02)
print("보낸 프레임:", f.hex(" ").upper())
print("받은 프레임:", bad.hex(" ").upper())
print("해석 결과  :", parse_frame(bad))

### 체크섬 vs CRC — 오류를 얼마나 잘 잡아낼까?
XOR 체크섬은 간단하지만, **같은 자리의 비트가 두 개 뒤집히면** 서로 상쇄되어 오류를 놓칩니다.
**CRC(순환중복검사)** 는 더 복잡한 나눗셈 원리로 훨씬 많은 오류를 잡아냅니다. 같은 형식에 CRC-16(2바이트)을 쓰는 프레임을 만들어 비교합니다.

In [ ]:
def build_frame_crc(sensor_id, seq, brix, temp_c):
    body = struct.pack(">BBHh", sensor_id, seq & 0xFF, round(brix * 10), round(temp_c * 10))
    crc = binascii.crc_hqx(body, 0xFFFF)            # CRC-16-CCITT
    return bytes([STX]) + body + struct.pack(">H", crc) + bytes([ETX])   # 총 10바이트

def parse_frame_crc(frame: bytes):
    if len(frame) != 10 or frame[0] != STX or frame[-1] != ETX:
        return None, "FRAME_ERR"
    body, (crc,) = frame[1:7], struct.unpack(">H", frame[7:9])
    if binascii.crc_hqx(body, 0xFFFF) != crc:
        return None, "CRC_ERR"
    sid, seq, b10, t10 = struct.unpack(">BBHh", body)
    return {"sensor_id": sid, "seq": seq, "brix": b10 / 10, "temp": t10 / 10}, "OK"

def ber_experiment(builder, parser, ber, trials=4000):
    corrupted = detected = missed = 0
    for k in range(trials):
        original = builder(3, k, round(random.uniform(8, 20), 1), round(random.uniform(15, 30), 1))
        received = noisy_channel(original, ber)
        if received == original:
            continue
        corrupted += 1
        data, status = parser(received)
        if status != "OK":
            detected += 1
        else:
            missed += 1          # 망가졌는데 정상으로 통과 = 가장 위험!
    return corrupted, detected, missed

random.seed(7)
rows = []
for ber in [0.001, 0.01, 0.03, 0.06]:
    for name, b, p in [("XOR 체크섬", build_frame, parse_frame), ("CRC-16", build_frame_crc, parse_frame_crc)]:
        c, d, m = ber_experiment(b, p, ber)
        rows.append({"BER": ber, "방식": name, "손상 프레임": c, "검출": d, "놓침(미검출)": m,
                     "놓침 비율(%)": round(m / c * 100, 2) if c else 0})
pd.DataFrame(rows)

### ✏️ 과제 3-1. 빠진 프레임 찾기
수신 측에 도착한 프레임들의 `seq` 번호 리스트가 아래와 같습니다. 중간에 **빠진 번호**를 찾는 함수 `find_missing(seqs)`를 작성하세요.
(`seq`는 1바이트라서 255 다음에 0으로 돌아간다는 점은 이번 과제에서는 무시합니다.)

In [ ]:
received_seqs = [10, 11, 12, 14, 15, 18, 19, 20]

def find_missing(seqs):
    # 여기에 코드를 작성하세요
    pass

# print(find_missing(received_seqs))   # 기대 결과: [13, 16, 17]

<details><summary>▶ 정답 보기</summary>

```python
def find_missing(seqs):
    expected = set(range(seqs[0], seqs[-1] + 1))
    return sorted(expected - set(seqs))
```
</details>

### 🤔 생각해 보기
- CRC 프레임은 1바이트가 더 깁니다. 전송 시간이 늘어나는 대신 얻는 것은?
- BER이 아주 커지면 XOR 체크섬의 놓침 비율이 오히려 줄어듭니다. STX·ETX 바이트도 함께 망가지기 때문인데, 이것이 좋은 소식일까요?
- 오류를 **검출**한 뒤에는 어떻게 해야 할까요? (재전송 요청 → 실습 5의 QoS로 이어집니다)

---
## 실습 4. 계층별 캡슐화 — 데이터가 인터넷을 타기까지

### 📖 개념
메인 보드는 측정값을 Wi-Fi로 인터넷의 서버(MQTT 브로커)에 보냅니다. 이때 데이터는 각 계층을 지나며 **헤더**가 하나씩 덧붙습니다(**캡슐화**).

```
응용 계층   :                                          [MQTT 헤더][  JSON 데이터  ]
전송 계층   :                                [TCP 헤더][MQTT 헤더][  JSON 데이터  ]
인터넷 계층 :                       [IP 헤더][TCP 헤더][MQTT 헤더][  JSON 데이터  ]
링크 계층   :            [Ethernet 헤더][IP 헤더][TCP 헤더][MQTT 헤더][  JSON 데이터  ][FCS]
```
헤더는 꼭 필요하지만, 진짜 데이터(페이로드)가 작으면 **헤더가 차지하는 비율(오버헤드)** 이 커집니다.

> 실제 Wi-Fi 프레임 헤더는 Ethernet보다 크고 구조도 다르지만, 여기서는 이해하기 쉽게 Ethernet 기준으로 계산합니다.

In [ ]:
reading = {"farm": "A", "fruit": "apple", "brix": 14.3, "temp": 21.5,
           "ts": "2026-09-15T10:30:00"}
payload = json.dumps(reading).encode("utf-8")
topic = "orchard/A/apple/brix"
print("JSON 문자열:", payload.decode())
print("페이로드 크기:", len(payload), "바이트")

In [ ]:
def encapsulate(payload_len, topic, qos=0):
    # 계층별 헤더 크기(바이트)를 딕셔너리로 반환
    mqtt = 2 + 2 + len(topic.encode()) + (2 if qos > 0 else 0)  # 고정헤더 + 토픽길이 + 토픽 + 패킷ID
    return {
        "Ethernet 헤더": 14,
        "IP 헤더": 20,
        "TCP 헤더": 20,
        "MQTT 헤더": mqtt,
        "페이로드(데이터)": payload_len,
        "Ethernet FCS": 4,
    }

def show_layers(layers):
    total = sum(layers.values())
    for name, size in layers.items():
        bar = "█" * max(1, round(size / total * 50))
        print(f"{name:<14} {size:>5} B  {bar}")
    eff = layers["페이로드(데이터)"] / total * 100
    print(f"{'합계':<14} {total:>5} B   → 전송 효율 {eff:.1f}%")
    return eff

eff_single = show_layers(encapsulate(len(payload), topic))

### 효율을 높이는 방법 비교
1. **JSON 1건씩** 보내기 (위의 방법)
2. **10건을 모아** JSON 배열 하나로 보내기 (배치 전송)
3. **바이너리**로 압축해서 보내기 (`struct` 사용, 실습 3의 프레임처럼)

In [ ]:
batch = json.dumps([reading] * 10).encode()
binary = struct.pack(">BHhI", 1, 143, 215, 1789437000)   # 농장번호, 당도×10, 온도×10, 시각(초)

cases = {
    "JSON x1": (len(payload), 1),
    "JSON batch x10": (len(batch), 10),
    "Binary x1": (len(binary), 1),
}
summary = []
for name, (size, n) in cases.items():
    layers = encapsulate(size, topic)
    total = sum(layers.values())
    summary.append({"방식": name, "측정 건수": n, "페이로드(B)": size,
                    "전체(B)": total, "효율(%)": round(size / total * 100, 1),
                    "1건당 전송량(B)": round(total / n, 1)})
df_eff = pd.DataFrame(summary)
display(df_eff)

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(df_eff["방식"], df_eff["1건당 전송량(B)"], color="white", edgecolor=INK, hatch="//")
ax.set_xlabel("Bytes on the wire per reading"); ax.set_title("Cost of sending one reading")
ax.invert_yaxis(); plt.show()

### 🤔 생각해 보기
- 바이너리는 효율(%)이 가장 낮게 나오는데도 1건당 전송량은 가장 적습니다. 두 지표 중 무엇을 기준으로 판단해야 할까요?
- 바이너리는 전송량이 가장 적은데 왜 많은 IoT 서비스가 JSON을 쓸까요? (사람이 읽기, 호환성, 확장성)
- 배치 전송의 단점은? (힌트: 경보가 늦게 도착한다면?)
- 배터리로 동작하는 센서라면 어떤 방식을 고르겠습니까?

---
## 실습 5. MQTT 발행/구독 — 브로커 직접 만들기

### 📖 개념
**MQTT**는 IoT에서 가장 많이 쓰는 메시지 프로토콜입니다. 센서와 앱이 서로 직접 연결하지 않고 **브로커**를 통해 주고받습니다.

```
 [센서 A: 사과] ──발행──┐                 ┌──▶ [대시보드]      구독: orchard/#
 [센서 B: 포도] ──발행──┼──▶ [ 브로커 ] ──┼──▶ [수확 경보]     구독: orchard/+/+/brix
 [센서 C: 딸기] ──발행──┘                 └──▶ [B농장 관리자]  구독: orchard/B/#
```
- **토픽(topic)**: 메시지 주소. `/`로 계층을 나눔 → `orchard/A/apple/brix`
- **와일드카드**: `+`는 한 단계, `#`은 그 아래 전부
- **QoS 0**: 한 번만 보냄(유실 가능) / **QoS 1**: 수신 확인(ACK)이 올 때까지 재전송(중복 가능)

In [ ]:
def topic_matches(topic_filter, topic):
    # 구독 필터가 토픽과 일치하는지 검사 (+, # 와일드카드 지원)
    f_parts, t_parts = topic_filter.split("/"), topic.split("/")
    for i, part in enumerate(f_parts):
        if part == "#":
            return True
        if i >= len(t_parts):
            return False
        if part != "+" and part != t_parts[i]:
            return False
    return len(f_parts) == len(t_parts)

tests = [("orchard/#", "orchard/A/apple/brix"),
         ("orchard/+/+/brix", "orchard/B/grape/brix"),
         ("orchard/B/#", "orchard/A/apple/brix"),
         ("orchard/+/brix", "orchard/A/apple/brix")]
for flt, t in tests:
    print(f"{flt:<20} vs {t:<24} → {topic_matches(flt, t)}")

### ✏️ 과제 5-1. 와일드카드 예측하기
코드를 실행하기 전에 결과를 먼저 예측해 보세요.
1. `orchard/+/apple/#` 와 `orchard/C/apple/brix`
2. `orchard/A` 와 `orchard/A/apple/brix`
3. `#` 와 `farm/test`

In [ ]:
# 예측한 뒤 실행해서 확인하세요
for flt, t in [("orchard/+/apple/#", "orchard/C/apple/brix"),
               ("orchard/A", "orchard/A/apple/brix"),
               ("#", "farm/test")]:
    print(f"{flt:<20} vs {t:<24} → {topic_matches(flt, t)}")

### 브로커와 불안정한 무선망 만들기
센서 → 브로커 구간이 무선이라서 일정 확률로 **패킷이 사라진다**고 가정합니다. 클래스 두 개를 만듭니다.
- `Network`: 손실률과 지연 시간을 흉내 냄
- `Broker`: 구독자 목록을 관리하고, 메시지가 오면 맞는 구독자에게 전달(콜백 호출)

In [ ]:
class Network:
    def __init__(self, loss_rate=0.1, base_ms=20, jitter_ms=30, timeout_ms=200):
        self.loss_rate = loss_rate
        self.base_ms, self.jitter_ms, self.timeout_ms = base_ms, jitter_ms, timeout_ms

    def transmit(self):
        # (전달 성공 여부, 걸린 시간 ms)
        ok = random.random() >= self.loss_rate
        return ok, self.base_ms + random.expovariate(1 / self.jitter_ms)


class Broker:
    def __init__(self, network):
        self.net = network
        self.subscriptions = []          # (구독자 이름, 토픽 필터, 콜백)
        self.stats = {"sent": 0, "delivered": 0, "lost": 0, "duplicates": 0, "latency": []}
        self.verbose = True

    def subscribe(self, name, topic_filter, callback):
        self.subscriptions.append((name, topic_filter, callback))
        if self.verbose:
            print(f"[브로커] '{name}' 구독 등록: {topic_filter}")

    def _deliver(self, topic, payload):
        for name, flt, cb in self.subscriptions:
            if topic_matches(flt, topic):
                cb(topic, payload)

    def publish(self, topic, payload, qos=0, max_retries=5):
        self.stats["sent"] += 1
        elapsed, arrived = 0.0, 0
        attempts = 1 if qos == 0 else max_retries
        for _ in range(attempts):
            ok, ms = self.net.transmit()               # 센서 → 브로커 (PUBLISH)
            if not ok:
                elapsed += self.net.timeout_ms         # 응답 없음 → 기다렸다가 재전송
                continue
            elapsed += ms
            arrived += 1
            self._deliver(topic, payload)
            if qos == 0:
                break
            ack_ok, ack_ms = self.net.transmit()       # 브로커 → 센서 (PUBACK)
            if ack_ok:
                elapsed += ack_ms
                break
            elapsed += self.net.timeout_ms             # ACK 유실 → 센서는 또 보냄(중복 발생)
        if arrived == 0:
            self.stats["lost"] += 1
        else:
            self.stats["delivered"] += 1
            self.stats["duplicates"] += arrived - 1
            self.stats["latency"].append(elapsed)

### 가상 농장 센서
세 농장에서 30일 동안 하루 8번씩 과일 당도를 측정합니다. 날이 갈수록 당도가 오르다가 일정 수준에서 멈추는 모습(S자 곡선)으로 흉내 냅니다.

In [ ]:
FARMS = {
    #  농장: (과일,        시작 당도, 최종 당도, 중간 도달일, 수확 기준 당도)
    "A": ("apple",      9.0, 15.5, 15, 14.0),
    "B": ("grape",     10.0, 19.0, 18, 16.0),
    "C": ("strawberry", 6.0, 12.0, 12, 10.0),
}
THRESHOLD = {fruit: th for fruit, _, _, _, th in FARMS.values()}
START = datetime(2026, 9, 1, 8, 0)

def simulate_brix(farm, day):
    fruit, lo, hi, mid, _ = FARMS[farm]
    trend = lo + (hi - lo) / (1 + math.exp(-(day - mid) / 3))   # S자 곡선
    true_brix = trend + random.gauss(0, 0.6)                     # 과일마다 차이
    return round(measure_brix(true_brix, noise=0.05), 1)         # 실습 2의 센서로 측정!

def sensor_messages(days=30, per_day=8):
    msg_id = 0
    for day in range(days):
        for k in range(per_day):
            for farm, (fruit, *_rest) in FARMS.items():
                msg_id += 1
                ts = START + timedelta(days=day, minutes=90 * k)
                payload = {"msg_id": msg_id, "farm": farm, "fruit": fruit, "day": day + 1,
                           "brix": simulate_brix(farm, day), "ts": ts.isoformat()}
                yield f"orchard/{farm}/{fruit}/brix", json.dumps(payload)

topic0, msg0 = next(sensor_messages())
print(topic0); print(msg0)

### 구독자(콜백 함수) 만들고 실행하기 — QoS 0

In [ ]:
def run_simulation(qos, loss_rate, verbose=True):
    random.seed(2026)
    broker = Broker(Network(loss_rate=loss_rate))
    dashboard, b_count, harvested = [], {"n": 0}, set()

    def on_dashboard(topic, payload):           # 모든 메시지 저장
        dashboard.append(json.loads(payload))

    def on_harvest_alarm(topic, payload):       # 기준 당도를 처음 넘으면 알림
        d = json.loads(payload)
        if d["brix"] >= THRESHOLD[d["fruit"]] and d["farm"] not in harvested:
            harvested.add(d["farm"])
            if verbose:
                print(f"  🔔 [경보] {d['day']}일차 {d['farm']}농장 {d['fruit']} "
                      f"당도 {d['brix']} ≥ 기준 {THRESHOLD[d['fruit']]}")

    def on_farm_b(topic, payload):              # B농장 메시지만 집계
        b_count["n"] += 1

    broker.verbose = verbose
    broker.subscribe("대시보드", "orchard/#", on_dashboard)
    broker.subscribe("수확 경보", "orchard/+/+/brix", on_harvest_alarm)
    broker.subscribe("B농장 관리자", "orchard/B/#", on_farm_b)

    for topic, payload in sensor_messages():
        broker.publish(topic, payload, qos=qos)

    s = broker.stats
    result = {"QoS": qos, "손실률": loss_rate, "발행": s["sent"], "도착": s["delivered"],
              "유실": s["lost"], "중복 수신": s["duplicates"],
              "평균 지연(ms)": float(round(np.mean(s["latency"]), 1)),
              "B농장 수신": b_count["n"]}
    return result, dashboard

r0, dash0 = run_simulation(qos=0, loss_rate=0.1)
r0

### QoS 1로 다시 실행하고 비교하기

In [ ]:
r1, dash1 = run_simulation(qos=1, loss_rate=0.1, verbose=False)
pd.DataFrame([r0, r1])

### ✏️ 과제 5-2. 중복 메시지 제거하기
QoS 1에서는 ACK가 유실되면 같은 메시지가 두 번 이상 도착합니다. `dash1` 리스트에서 `msg_id`가 같은 메시지를 하나만 남기는 함수 `dedupe(messages)`를 작성하세요.

In [ ]:
def dedupe(messages):
    # 여기에 코드를 작성하세요
    pass

# clean = dedupe(dash1)
# print(len(dash1), "→", len(clean))

<details><summary>▶ 정답 보기</summary>

```python
def dedupe(messages):
    seen, result = set(), []
    for m in messages:
        if m["msg_id"] not in seen:
            seen.add(m["msg_id"])
            result.append(m)
    return result
```
</details>

In [ ]:
# 정답 코드 (실습 6에서 사용)
def dedupe(messages):
    seen, result = set(), []
    for m in messages:
        if m["msg_id"] not in seen:
            seen.add(m["msg_id"])
            result.append(m)
    return result

clean = dedupe(dash1)
print(f"수신 {len(dash1)}건 → 중복 제거 후 {len(clean)}건")

In [ ]:
# 실험: 손실률이 커질 때 QoS 0과 QoS 1의 차이
rows = []
for loss in [0.0, 0.05, 0.1, 0.2, 0.3]:
    for q in [0, 1]:
        r, _ = run_simulation(qos=q, loss_rate=loss, verbose=False)
        rows.append(r)
df_qos = pd.DataFrame(rows)
display(df_qos[["QoS", "손실률", "유실", "중복 수신", "평균 지연(ms)"]])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for q, style in [(0, "-o"), (1, "--s")]:
    d = df_qos[df_qos["QoS"] == q]
    axes[0].plot(d["손실률"], d["유실"] / d["발행"] * 100, style, color=INK, mfc="white", label=f"QoS {q}")
    axes[1].plot(d["손실률"], d["평균 지연(ms)"], style, color=INK, mfc="white", label=f"QoS {q}")
axes[0].set(title="Messages lost (%)", xlabel="Packet loss rate")
axes[1].set(title="Average delay (ms)", xlabel="Packet loss rate")
for a in axes: a.legend()
plt.tight_layout(); plt.show()

### 🤔 생각해 보기
- QoS 1은 신뢰성을 얻는 대신 무엇을 대가로 치르나요?
- **수확 경보**처럼 중요한 메시지와 **5분마다 오는 온도값**에는 각각 어떤 QoS가 어울릴까요?
- MQTT에는 정확히 한 번만 전달하는 **QoS 2**도 있습니다. 어떤 방법으로 중복을 막을지 추측해 보세요.

---
## 실습 6. 수집 데이터 분석 — 언제 수확할까?

브로커를 통해 모은 데이터(중복 제거한 `clean`)를 `pandas`로 분석합니다.

In [ ]:
df = pd.DataFrame(clean)
df["ts"] = pd.to_datetime(df["ts"])
print("데이터 크기:", df.shape)
df.head()

In [ ]:
# 농장별 기초 통계
df.groupby(["farm", "fruit"])["brix"].describe().round(2)

In [ ]:
# 날짜별 평균 당도와 수확 가능 첫날
daily = df.groupby(["farm", "fruit", "day"])["brix"].mean().reset_index()

harvest = []
for (farm, fruit), g in daily.groupby(["farm", "fruit"]):
    ready = g[g["brix"] >= THRESHOLD[fruit]]
    harvest.append({"농장": farm, "과일": fruit, "기준 당도": THRESHOLD[fruit],
                    "수확 가능 첫날": int(ready["day"].min()) if len(ready) else "미도달",
                    "마지막날 평균": round(g["brix"].iloc[-1], 2)})
pd.DataFrame(harvest)

In [ ]:
styles = {"A": ("-", "o"), "B": ("--", "s"), "C": (":", "^")}
fig, ax = plt.subplots(figsize=(9, 4.5))
for (farm, fruit), g in daily.groupby(["farm", "fruit"]):
    ls, mk = styles[farm]
    ax.plot(g["day"], g["brix"], ls, marker=mk, ms=4, color=INK, mfc="white", label=f"Farm {farm} ({fruit})")
    ax.axhline(THRESHOLD[fruit], color=GRAY, lw=0.8, ls=ls)
    ax.text(30.4, THRESHOLD[fruit], f"{fruit} {THRESHOLD[fruit]}", va="center", fontsize=8, color=GRAY)
ax.set_xlabel("Day"); ax.set_ylabel("Mean sugar content (°Brix)")
ax.set_title("Daily mean Brix by farm (gray lines = harvest threshold)")
ax.set_xlim(0, 34); ax.legend(loc="upper left")
plt.show()

### ✏️ 과제 6-1. 측정 시간대별 분석
`df["ts"].dt.hour`로 측정 시각(시)을 꺼낼 수 있습니다. **농장별·시간대별 평균 당도**를 표로 만들어 보세요. (힌트: `pivot_table`)

In [ ]:
# 여기에 코드를 작성하세요

<details><summary>▶ 정답 보기</summary>

```python
df["hour"] = df["ts"].dt.hour
df.pivot_table(index="hour", columns="farm", values="brix", aggfunc="mean").round(2)
```
이 시뮬레이션은 시간대 효과를 넣지 않았으므로 큰 차이가 없어야 정상입니다. 만약 차이가 보인다면 우연(잡음)입니다.
</details>

In [ ]:
# 수집 데이터를 CSV로 저장 (Colab 왼쪽 파일 탭에서 내려받기 가능)
df.to_csv("brix_data.csv", index=False, encoding="utf-8-sig")
print("brix_data.csv 저장 완료")

---
## 실습 7. (선택) 진짜 MQTT 브로커에 연결하기

지금까지 만든 브로커 대신 인터넷의 **공개 MQTT 브로커**를 사용합니다. 옆자리 친구와 같은 토픽을 정하면 서로의 메시지를 받을 수 있습니다.

> ⚠️ **보안 주의**
> - 공개 브로커의 1883 포트는 **암호화되지 않습니다.** 토픽만 알면 전 세계 누구나 메시지를 볼 수 있습니다.
> - 이름, 전화번호 같은 **개인정보는 절대 보내지 마세요.**
> - 실제 서비스는 **TLS(8883 포트)** 와 **아이디·비밀번호 인증**을 사용합니다. Wireshark로 1883 패킷을 잡으면 내용이 그대로 보인다는 것을 보안 수업에서 확인해 보세요.
> - 학교·기관 네트워크에서 1883 포트가 막혀 있으면 연결되지 않을 수 있습니다.

아래 첫 셀의 주석을 풀어 라이브러리를 설치하고, 두 번째 셀에서 `RUN_REAL_MQTT = True`, `MY_ID`를 본인 번호로 바꾼 뒤 실행하세요.

In [ ]:
# !pip -q install paho-mqtt

In [ ]:
RUN_REAL_MQTT = False          # True로 바꾸면 실행
MY_ID = "student01"            # 본인 번호로 변경 (예: student07)
BROKER_HOST, BROKER_PORT = "broker.hivemq.com", 1883
REAL_TOPIC = f"kdt-brix-lab/{MY_ID}/brix"

if RUN_REAL_MQTT:
    import time
    import paho.mqtt.client as mqtt

    inbox = []

    def on_connect(client, userdata, flags, reason_code, properties):
        print("연결 결과:", reason_code)
        client.subscribe(REAL_TOPIC, qos=1)

    def on_message(client, userdata, msg):
        data = json.loads(msg.payload)
        inbox.append(data)
        print(f"  수신 ← {msg.topic}: {data}")

    try:
        client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2,
                             client_id=f"{MY_ID}-{random.randint(1000, 9999)}")
        client.on_connect, client.on_message = on_connect, on_message
        client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
        client.loop_start()
        time.sleep(2)
        for i in range(5):
            data = {"msg_id": i + 1, "farm": "LAB", "fruit": "apple",
                    "brix": round(measure_brix(random.uniform(12, 15)), 1)}
            client.publish(REAL_TOPIC, json.dumps(data), qos=1)
            print(f"발행 → {REAL_TOPIC}: {data}")
            time.sleep(1)
        time.sleep(2)
        client.loop_stop(); client.disconnect()
        print(f"\n보낸 5건 중 {len(inbox)}건을 다시 받았습니다.")
    except Exception as e:
        print("연결 실패:", e, "\n네트워크에서 1883 포트가 막혀 있을 수 있습니다.")
else:
    print("RUN_REAL_MQTT = False → 건너뜀")

---
## 🏁 정리

| 단계 | 우리가 만든 것 | 핵심 정보통신 개념 |
|---|---|---|
| 1 | `brix_to_n`, `pixel_to_brix` | 센서는 물리량을 전기신호로 바꾼다. 분해능은 센서 구조가 정한다 |
| 2 | `adc`, `measure_brix` | 양자화, 잡음, 평균화로 SNR 개선 |
| 3 | `build_frame`, `parse_frame` | 프레임 구조, bps, 오류 검출(체크섬 < CRC) |
| 4 | `encapsulate` | 계층별 헤더, 오버헤드, 효율 |
| 5 | `Broker`, `Network` | 발행/구독, 토픽, QoS의 신뢰성–지연 트레이드오프 |
| 6 | pandas 분석 | 수집된 데이터에서 의사결정(수확 시기) 도출 |

## 🚀 도전 과제
1. **온도 보정(ATC)**: 실제 굴절률은 온도가 1°C 오를 때 약 0.0001 줄어듭니다. 프레임의 `TEMP` 값을 이용해 20°C 기준으로 당도를 보정하는 함수를 만들어 보세요.
2. **더 똑똑한 경계 검출**: 실습 2에서 잡음이 클 때 생기는 치우침을 줄이려면? (이동평균, 밝기 변화가 가장 큰 지점 찾기)
3. **seq 순환 처리**: 1바이트 `seq`는 255 다음이 0입니다. 과제 3-1의 `find_missing`이 이를 처리하도록 고쳐 보세요.
4. **바이너리 MQTT**: 실습 5의 페이로드를 JSON 대신 `struct` 바이너리로 바꾸고, 실습 4의 효율 계산으로 비교해 보세요.
5. **배터리 계산**: 센서가 메시지 1건 보낼 때 0.05mAh를 쓴다면, 2000mAh 배터리로 QoS 0과 QoS 1(손실률 20%)에서 각각 며칠 버틸까요?

## 📝 복습 질문
1. 8비트 ADC는 몇 단계로 전압을 표현하나요?
2. 115200bps로 9바이트 UART 프레임을 보내는 데 걸리는 시간은?
3. XOR 체크섬이 오류를 놓치는 대표적인 경우는?
4. 토픽 필터 `orchard/+/apple/brix`가 받을 수 있는 토픽의 예를 두 개 쓰세요.
5. QoS 1에서 중복 메시지가 생기는 이유는?

<details><summary>▶ 복습 질문 정답</summary>

1. 2⁸ = 256단계
2. 9 × 10비트 = 90비트, 90 / 115200 ≈ 0.78ms
3. 같은 비트 위치에서 비트 2개(짝수 개)가 뒤집힌 경우
4. `orchard/A/apple/brix`, `orchard/Z/apple/brix` 등
5. 메시지는 도착했지만 수신 확인(PUBACK)이 유실되어 송신 측이 같은 메시지를 다시 보내기 때문
</details>